In [10]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
df = pd.read_csv("garments_worker_productivity.csv")
print(df.head())

       date   quarter  department       day  team  targeted_productivity  \
0  1/1/2015  Quarter1      sweing  Thursday     8                   0.80   
1  1/1/2015  Quarter1  finishing   Thursday     1                   0.75   
2  1/1/2015  Quarter1      sweing  Thursday    11                   0.80   
3  1/1/2015  Quarter1      sweing  Thursday    12                   0.80   
4  1/1/2015  Quarter1      sweing  Thursday     6                   0.80   

     smv     wip  over_time  incentive  idle_time  idle_men  \
0  26.16  1108.0       7080         98        0.0         0   
1   3.94     NaN        960          0        0.0         0   
2  11.41   968.0       3660         50        0.0         0   
3  11.41   968.0       3660         50        0.0         0   
4  25.90  1170.0       1920         50        0.0         0   

   no_of_style_change  no_of_workers  actual_productivity  
0                   0           59.0             0.940725  
1                   0            8.0        

In [11]:
df.columns = df.columns.str.strip()
# We are taking the 'actual productivity' as the Target => Regression.
y_reg = df['actual_productivity']

# we did not have any Target column for Classification so we came up with one.
df['Meets_Target'] = (df['actual_productivity'] >= df['targeted_productivity']).astype(int)
y_clf = df['Meets_Target']

In [12]:
X = df.drop(columns=['actual_productivity', 'Meets_Target', 'date'])
num_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ])

X_processed = preprocessor.fit_transform(X)

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X_processed, y_reg, y_clf, test_size=0.2, random_state=42
)

print("--- Part A: Scikit-Learn Implementation ---")

lin_reg = LinearRegression()

start_train_reg = time.time()
lin_reg.fit(X_train, y_reg_train)
end_train_reg = time.time()
train_time_reg = end_train_reg - start_train_reg

start_pred_reg = time.time()
y_reg_pred = lin_reg.predict(X_test)
end_pred_reg = time.time()
pred_time_reg = end_pred_reg - start_pred_reg

mae = mean_absolute_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
r2 = r2_score(y_reg_test, y_reg_pred)

print("Linear Regression Results:")
print(f"Training Time:   {train_time_reg:.6f} seconds")
print(f"Prediction Time: {pred_time_reg:.6f} seconds")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}\n")

log_reg = LogisticRegression(max_iter=1000)

start_train_clf = time.time()
log_reg.fit(X_train, y_clf_train)
end_train_clf = time.time()
train_time_clf = end_train_clf - start_train_clf

start_pred_clf = time.time()
y_clf_pred = log_reg.predict(X_test)
end_pred_clf = time.time()
pred_time_clf = end_pred_clf - start_pred_clf

accuracy = accuracy_score(y_clf_test, y_clf_pred)
precision = precision_score(y_clf_test, y_clf_pred)
recall = recall_score(y_clf_test, y_clf_pred)
f1 = f1_score(y_clf_test, y_clf_pred)

print("Logistic Regression Results:")
print(f"Training Time:   {train_time_clf:.6f} seconds")
print(f"Prediction Time: {pred_time_clf:.6f} seconds")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

--- Part A: Scikit-Learn Implementation ---
Linear Regression Results:
Training Time:   0.029596 seconds
Prediction Time: 0.001995 seconds
MAE:  0.1085
RMSE: 0.1486
R2:   0.1682

Logistic Regression Results:
Training Time:   0.035419 seconds
Prediction Time: 0.000997 seconds
Accuracy:  0.7750
Precision: 0.7808
Recall:    0.9661
F1-Score:  0.8636


In [ ]:
df = pd.read_csv("garments_worker_productivity.csv")
df.columns = df.columns.str.strip()

y_reg_raw = df['actual_productivity'].values
y_clf_raw = (df['actual_productivity'] >= df['targeted_productivity']).astype(int).values
X_raw = df.drop(columns=['actual_productivity', 'date'])

np.random.seed(42)
indices = np.random.permutation(len(X_raw))
test_size = int(0.2 * len(X_raw))
test_idx, train_idx = indices[:test_size], indices[test_size:]

X_train_raw = X_raw.iloc[train_idx].copy()
X_test_raw = X_raw.iloc[test_idx].copy()
y_reg_train = y_reg_raw[train_idx]
y_reg_test = y_reg_raw[test_idx]
y_clf_train = y_clf_raw[train_idx]
y_clf_test = y_clf_raw[test_idx]

num_cols = X_train_raw.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train_raw.select_dtypes(include=['object']).columns

# A. Missing Value Handling (Fit on train, apply to both)
medians = X_train_raw[num_cols].median()
X_train_raw[num_cols] = X_train_raw[num_cols].fillna(medians)
X_test_raw[num_cols] = X_test_raw[num_cols].fillna(medians)

X_train_raw[cat_cols] = X_train_raw[cat_cols].fillna('missing')
X_test_raw[cat_cols] = X_test_raw[cat_cols].fillna('missing')

# B. Categorical Encoding (One-Hot Encoding via pandas)
X_train_ohe = pd.get_dummies(X_train_raw, columns=cat_cols)
X_test_ohe = pd.get_dummies(X_test_raw, columns=cat_cols)
# Align columns to ensure train and test have the exact same dummy features
X_train_ohe, X_test_ohe = X_train_ohe.align(X_test_ohe, join='left', axis=1, fill_value=0)

# C. Feature Scaling (Standardization: (X - mean) / std)
means = X_train_ohe[num_cols].mean()
stds = X_train_ohe[num_cols].std(ddof=0)
stds[stds == 0] = 1 # Prevent division by zero

X_train_ohe[num_cols] = (X_train_ohe[num_cols] - means) / stds
X_test_ohe[num_cols] = (X_test_ohe[num_cols] - means) / stds

# Convert to NumPy arrays and add bias term (column of 1s) for the models
X_train_mat = X_train_ohe.to_numpy(dtype=float)
X_test_mat = X_test_ohe.to_numpy(dtype=float)

X_train_mat = np.c_[np.ones(X_train_mat.shape[0]), X_train_mat]
X_test_mat = np.c_[np.ones(X_test_mat.shape[0]), X_test_mat]

def calc_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def calc_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

def calc_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return 1 - (ss_res / ss_tot)

def calc_clf_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return accuracy, precision, recall, f1

print("--- PART B: FROM-SCRATCH IMPLEMENTATION ---\n")

class ManualLinearRegression:
    def fit(self, X, y):
        # Using pseudo-inverse for numerical stability
        self.theta = np.linalg.pinv(X.T @ X) @ X.T @ y
        
    def predict(self, X):
        return X @ self.theta

manual_lin_reg = ManualLinearRegression()

start_train_reg = time.time()
manual_lin_reg.fit(X_train_mat, y_reg_train)
train_time_reg = time.time() - start_train_reg

start_pred_reg = time.time()
y_reg_pred_manual = manual_lin_reg.predict(X_test_mat)
pred_time_reg = time.time() - start_pred_reg

print("Manual Linear Regression Results:")
print(f"Training Time:   {train_time_reg:.6f} seconds")
print(f"Prediction Time: {pred_time_reg:.6f} seconds")
print(f"MAE:  {calc_mae(y_reg_test, y_reg_pred_manual):.4f}")
print(f"RMSE: {calc_rmse(y_reg_test, y_reg_pred_manual):.4f}")
print(f"R2:   {calc_r2(y_reg_test, y_reg_pred_manual):.4f}\n")

class ManualLogisticRegression:
    def __init__(self, lr=0.1, max_iter=2000):
        self.lr = lr
        self.max_iter = max_iter
        
    def sigmoid(self, z):
        # Clip z to prevent np.exp overflow
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))
        
    def fit(self, X, y):
        m, n = X.shape
        self.theta = np.zeros(n)
        for _ in range(self.max_iter):
            h = self.sigmoid(X @ self.theta)
            gradient = (1 / m) * X.T @ (h - y)
            self.theta -= self.lr * gradient
            
    def predict_proba(self, X):
        return self.sigmoid(X @ self.theta)
        
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

manual_log_reg = ManualLogisticRegression(lr=0.1, max_iter=2000)

start_train_clf = time.time()
manual_log_reg.fit(X_train_mat, y_clf_train)
train_time_clf = time.time() - start_train_clf

start_pred_clf = time.time()
y_clf_pred_manual = manual_log_reg.predict(X_test_mat)
pred_time_clf = time.time() - start_pred_clf

acc, prec, rec, f1_m = calc_clf_metrics(y_clf_test, y_clf_pred_manual)

print("Manual Logistic Regression Results:")
print(f"Training Time:   {train_time_clf:.6f} seconds")
print(f"Prediction Time: {pred_time_clf:.6f} seconds")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1_m:.4f}")

--- PART B: FROM-SCRATCH IMPLEMENTATION ---

Manual Linear Regression Results:
Training Time:   0.013586 seconds
Prediction Time: 0.000000 seconds
MAE:  0.1087
RMSE: 0.1489
R2:   0.1681

Manual Logistic Regression Results:
Training Time:   0.329504 seconds
Prediction Time: 0.000000 seconds
Accuracy:  0.7741
Precision: 0.7798
Recall:    0.9659
F1-Score:  0.8629
